## Introduction

In this assessment, a breast cancer dataset is analysed using a machine learning approach to classify cases as benign or malignant.
The main purpose of this study is to build a classification model and evaluate its performance, while also considering the medical impact of the results.

In particular, special attention is given to the risk of misclassifying malignant cases, as this can have serious consequences in real medical settings.
For this reason, the evaluation focuses not only on overall accuracy but also on clinically relevant metrics.

Logistic Regression is used as the baseline model in this analysis because it is easy to understand, widely used, and suitable for binary classification problems in healthcare.
The analysis follows the same steps covered in the lectures, including data preprocessing, feature scaling, model training, and performance evaluation

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.__version__, np.__version__

In [ ]:
df = pd.read_csv("MS4S16_Dataset.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df['diagnosis'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
numeric_cols = df.select_dtypes(include=['float64']).columns

In [ ]:
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

In [ ]:
df.isnull().sum().sum()

In [ ]:
df = df.dropna(subset=['diagnosis'])

In [ ]:
df.isnull().sum().sum()

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

In [ ]:
df['diagnosis'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

class_counts = df['diagnosis'].value_counts()

plt.figure(figsize=(5,4))
class_counts.plot(kind='bar')
plt.xlabel('Diagnosis (0 = Benign, 1 = Malignant)')
plt.ylabel('Number of Samples')
plt.title('Class Distribution of Diagnosis')
plt.show()

### Class Distribution Analysis

The dataset contains more benign cases than malignant ones, which means the classes are not perfectly balanced.
This situation is quite common in medical datasets and can affect how model performance is interpreted.

To make sure that both classes are represented fairly in the training and test sets, stratified sampling was applied during the train–test split.
This helps preserve the original class distribution and leads to a more reliable evaluation of the model.

In [ ]:
X = df.drop(columns=['id', 'diagnosis'])
y = df['diagnosis']

In [ ]:
X.columns

In [ ]:
X.shape, y.shape

### Handling Class Imbalance

During the exploratory data analysis, it was observed that benign cases are more frequent than malignant cases in the dataset.
Such class imbalance is common in medical classification problems and can influence model evaluation if not handled carefully.

To address this, stratified sampling was used when splitting the data into training and test sets.
This ensures that both sets reflect the original class distribution, leading to more reliable and fair performance evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

### Model Selection

Logistic Regression was selected as the classification model for this task because it is relatively simple and easy to interpret.
In medical applications, being able to understand a model’s behaviour is important, especially when decisions may affect patient outcomes.

Another reason for choosing Logistic Regression is that it works well for binary classification problems and provides probability estimates rather than only class labels.
These probability outputs are useful for further evaluation, such as analysing the model’s performance using ROC–AUC.

### Comparison of Classification Techniques

Several classification algorithms can be applied to medical diagnosis problems such as breast cancer detection.
In addition to Logistic Regression, commonly used techniques include Support Vector Machines (SVM) and ensemble-based methods such as Random Forests.

Support Vector Machines are effective in handling high-dimensional data and can model complex decision boundaries.
However, they are generally less interpretable, which can be a limitation in medical applications where transparency is important.

Random Forest classifiers often achieve high predictive performance by combining multiple decision trees.
Despite their strong accuracy, they can behave as black-box models, making it difficult to explain individual predictions.

In this assessment, Logistic Regression was selected as the primary model due to its interpretability, simplicity, and suitability for medical decision-support systems.
While other techniques may achieve slightly higher performance, the ability to clearly understand and explain model behaviour was prioritised.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled.shape, X_test_scaled.shape

In [ ]:
X_train_scaled.mean(), X_train_scaled.std()

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

# Apply PCA to the scaled training data
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)

# Explained variance ratio
explained_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(6,4))
plt.plot(explained_variance)
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance')
plt.grid(True)
plt.show()

### Principal Component Analysis (PCA)

Principal Component Analysis was used to reduce the number of features in the dataset while keeping most of the important information.
The explained variance plot indicates that a relatively small number of principal components explain a large proportion of the total variance.

This suggests that the dataset can be represented in a lower-dimensional space without losing much information, which is helpful for visualisation and further unsupervised analysis.

In [ ]:
pca_2 = PCA(n_components=2)
X_train_pca_2 = pca_2.fit_transform(X_train_scaled)
X_test_pca_2 = pca_2.transform(X_test_scaled)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters_train = kmeans.fit_predict(X_train_pca_2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
plt.scatter(X_train_pca_2[:, 0], X_train_pca_2[:, 1], c=clusters_train)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (k=2) on PCA Projection")
plt.show()

### Unsupervised Learning Interpretation

The PCA-based visualisation suggests that the data points can be grouped into two clusters using KMeans.
Although some separation between the clusters can be observed, there is also noticeable overlap, which is expected when working with real medical data.

Since this is an unsupervised method, the clusters do not directly represent benign or malignant cases.
However, the analysis still provides a useful overview of the structure of the dataset and helps support the results obtained from the supervised classification model.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

### Model Evaluation (Medical Perspective)

In medical diagnosis, not all prediction errors have the same impact.
In particular, false negatives are especially serious, as failing to identify a malignant tumour may delay treatment and negatively affect patient outcomes.

Based on the confusion matrix results, the model produces a relatively low number of false negative predictions.
This suggests that most malignant cases are correctly identified, which is an important consideration in medical classification tasks.

While the model is not flawless, the results indicate that it could be used as a supportive tool to assist clinicians in decision-making, rather than as a replacement for professional medical judgement.

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test_scaled)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc

### Model Performance Overview

In addition to accuracy, confusion matrix, and classification metrics, it is useful to examine how the model behaves across different classification thresholds.
This provides a more complete view of the model’s performance.

To do this, the Receiver Operating Characteristic (ROC) curve and the Area Under the Curve (AUC) score are used.
These metrics help show how well the model can distinguish between benign and malignant cases without relying on a single fixed threshold.

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

### ROC–AUC Analysis

The ROC–AUC score obtained is relatively high, indicating that the model is able to distinguish between benign and malignant cases reasonably well.
This suggests that the classifier maintains good performance across different decision thresholds, rather than being dependent on a single cut-off value.


### Supervised Model Comparison: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf)
rf_report = classification_report(y_test, y_pred_rf)

rf_accuracy

The Random Forest model achieved higher overall accuracy compared to Logistic Regression.
However, Logistic Regression remains more interpretable, which can be preferable in medical decision-support settings.

In [ ]:
rf_cm

In [ ]:
print(rf_report)

## Ethical Considerations and Limitations

### Ethical Considerations

The use of machine learning models in medical diagnosis raises important ethical concerns.
While the model developed in this analysis shows strong performance, it should not be used as a standalone diagnostic tool.
Incorrect predictions, particularly false negatives, could delay treatment and have serious consequences for patients.
Therefore, such models should only be used as decision-support tools alongside clinical expertise.

### Limitations

- The model was trained using a single dataset, which may limit its generalisability to other populations or clinical settings.
- Although multiple classification models were evaluated, only a limited number of techniques were explored in this assessment. 
Further work could include a broader comparison of additional models and more extensive hyperparameter tuning.
- Feature measurements are derived from processed medical images and may not fully represent real-world diagnostic complexity.

### Recommendations for Future Work

- Future studies could explore additional classification models and compare their performance.
- Incorporating cross-validation and hyperparameter tuning may further improve model robustness.
- Using larger and more diverse datasets could help improve generalisation and reduce potential bias.

### Conclusion

In this assessment, supervised machine learning models, including Logistic Regression and Random Forest, were applied to classify breast cancer cases as either benign or malignant.The analysis followed the general workflow covered in the lectures, including data preprocessing, feature scaling, model training, and evaluation.

Overall, the model shows good performance, with high accuracy and a strong ROC–AUC score.
From a medical perspective, the relatively low number of false negatives is particularly important, as it helps reduce the risk of failing to identify malignant cases.